# 03 — Modeling, Evaluation of deep neural network

This is the Phase 3/4 core of the project: train multiple classical ML models on the
TEP fault diagnosis task, evaluate them properly, tune the best one, and save it.

Task: **multi-class classification using deep neural network** — predict `faultNumber` (0 = normal, 1–20 = fault
type) from the 52 process sensor readings.


In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
#  Splits the dataset into training and testing sets to evaluate model performance.

from sklearn.preprocessing import LabelEncoder, StandardScaler
# LabelEncoder: Converts categorical labels (e.g., "cat", "dog") into numeric values.
# StandardScaler: Normalizes/standardizes numerical features so they have mean=0 and variance=1.

from sklearn.linear_model import Perceptron
# A simple linear classifier (single-layer neural network). Good for binary classification tasks.

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
# Lets you build a neural network layer-by-layer in a linear fashion.

from tensorflow.keras.layers import Dense
# Fully connected layer (every neuron connects to every neuron in the next layer).

from tensorflow.keras.layers import Dropout
# Randomly drops some neurons during training to prevent overfitting.

from tensorflow.keras.utils import to_categorical


In [38]:
path = '/content/drive/MyDrive/Sensornet.ai/data/raw/'   # your folder

# If your data is CSV
train = pd.read_csv(path + 'TEP_Random_Sample_5_percent.csv')
test  = pd.read_csv(path + 'TEP_Random_Testing_Sample_5_percent.csv')
print(train.shape, test.shape)

(262500, 55) (504000, 55)


In [39]:
train

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,12,274,3,0.25179,3666.0,4504.6,9.2813,27.034,42.061,2706.3,...,54.036,24.527,61.102,22.329,39.572,40.280,43.260,47.718,40.968,19.798
1,2,135,484,0.27358,3772.2,4671.0,9.5640,27.014,43.144,2693.4,...,55.508,27.068,65.509,21.172,81.806,39.661,46.617,24.117,42.513,15.949
2,9,324,428,0.24385,3649.0,4440.9,9.2751,26.854,42.291,2696.1,...,53.062,23.795,62.079,22.194,40.247,38.278,46.453,49.746,40.923,15.984
3,12,448,39,0.21421,3689.8,4530.7,9.2022,27.006,42.251,2722.8,...,54.890,21.343,60.005,22.629,38.644,44.087,43.871,49.086,42.211,20.310
4,7,74,50,0.34507,3633.9,4332.4,9.9731,26.553,42.639,2681.9,...,52.248,33.714,79.895,21.320,46.094,37.561,46.762,57.477,42.354,19.997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262495,0,385,156,0.25913,3730.5,4546.4,9.3020,26.581,42.460,2701.8,...,54.169,25.806,61.724,22.047,38.262,40.146,46.146,55.182,41.488,19.435
262496,20,360,65,0.30066,3655.5,4476.5,9.2364,26.935,42.268,2702.8,...,53.912,29.493,60.451,21.717,39.546,35.258,49.565,44.593,40.948,18.091
262497,0,275,140,0.28309,3683.6,4527.3,9.2719,26.751,42.433,2701.6,...,54.249,27.890,60.880,21.686,41.463,34.265,44.044,48.309,40.725,19.161
262498,11,286,136,0.24087,3743.5,4524.7,9.2256,26.945,42.308,2707.0,...,53.343,23.425,61.046,21.524,39.838,38.873,48.184,45.908,42.555,19.647


In [40]:
test

,faultNumber,simulationRun,sample,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,xmeas_7,...,xmv_2,xmv_3,xmv_4,xmv_5,xmv_6,xmv_7,xmv_8,xmv_9,xmv_10,xmv_11
0,5,83,120,0.256870,3651.8,4462.2,9.3350,26.872,42.342,2710.5,...,54.563,25.775,61.981,22.148000,40.127,40.621,45.641,49.02900,41.102,19.221
1,13,106,935,0.211550,3624.5,4645.8,9.3730,26.794,42.202,2687.8,...,55.790,20.756,61.992,22.418000,39.475,42.738,42.716,29.09800,41.805,18.575
2,4,409,585,0.199850,3692.0,4545.3,9.2944,26.839,42.189,2716.7,...,54.354,19.875,62.156,23.129000,39.210,41.365,52.337,46.68100,45.674,19.235
3,15,460,858,0.274550,3648.3,4564.3,9.3293,26.951,42.317,2709.6,...,54.226,27.038,63.001,21.618000,40.121,40.008,43.444,43.19700,40.390,16.098
4,8,285,403,0.346420,3704.7,4598.6,8.7290,27.007,42.049,2723.8,...,54.774,33.868,57.092,23.264000,50.242,36.738,44.026,22.52500,39.813,19.108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503995,16,86,799,0.254020,3653.6,4462.3,9.3875,27.307,42.580,2708.4,...,53.419,24.769,59.657,22.562000,40.281,41.603,45.251,47.72300,40.930,18.940
503996,4,132,872,0.241700,3639.2,4555.5,9.4651,26.718,42.390,2707.3,...,53.375,24.111,59.673,22.474000,41.409,32.700,48.946,45.75500,44.769,17.819
503997,11,488,615,0.279210,3705.4,4538.8,9.3406,26.870,42.400,2702.0,...,53.922,27.621,59.917,21.414000,42.314,36.463,50.945,47.58000,46.074,17.279
503998,0,415,129,0.246620,3623.8,4497.8,9.4867,26.631,42.394,2698.3,...,53.519,24.230,61.846,22.454000,40.195,38.658,42.409,55.73400,40.755,21.022


In [41]:
train['faultNumber'].value_counts()

,count
faultNumber,
11,12635
10,12626
20,12612
13,12612
8,12604
9,12572
0,12560
16,12553
14,12526


In [42]:
x_train = train.drop(columns=['faultNumber', 'sample', 'simulationRun'])
y_train = train['faultNumber']
x_test  = test.drop(columns=['faultNumber', 'sample', 'simulationRun'])
y_test  = test['faultNumber']

In [43]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled  = scaler.transform(x_test)

In [44]:
X_train_scaled

array([[-0.06128469,  0.05060256, -0.014111  , ..., -0.1626076 ,
        -0.09517801,  0.19607754],
       [ 0.08807247,  2.53157335,  1.51800335, ..., -1.53747487,
         0.06432678, -0.56515153],
       [-0.11570854, -0.34653966, -0.60062353, ..., -0.04446723,
        -0.09982378, -0.55822946],
       ...,
       [ 0.15325771,  0.46176156,  0.19489739, ..., -0.12817912,
        -0.12026517,  0.07009601],
       [-0.13613462,  1.86110384,  0.1709581 , ..., -0.26804846,
         0.06866283,  0.16621379],
       [ 0.09341889, -1.4001346 , -0.74517999, ...,  0.0680807 ,
        -0.07638845, -0.2184551 ]])

In [45]:
X_test_scaled

array([[-0.02646439, -0.281128  , -0.40450552, ..., -0.08623579,
        -0.08134394,  0.0819624 ],
       [-0.33710533, -0.91889168,  1.28597642, ..., -1.24730861,
        -0.00876668, -0.04579909],
       [-0.41730169,  0.65799654,  0.36063091, ..., -0.22301764,
         0.39066634,  0.08473123],
       ...,
       [ 0.12666268,  0.97103805,  0.30078269, ..., -0.17064673,
         0.43196208, -0.30211316],
       [-0.09672188, -0.93524459, -0.07672144, ...,  0.30436143,
        -0.11716799,  0.43815195],
       [-1.77279703,  3.40995849,  2.97553762, ..., -2.9539377 ,
         6.0013117 ,  5.38960194]])

In [46]:
y_train_cat = to_categorical(y_train,num_classes = 21)
y_test_cat = to_categorical(y_test,num_classes = 21)

In [47]:
y_train_cat

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [48]:
model = keras.Sequential([
    layers.Input(shape=(52,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(21, activation='softmax')
])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

In [52]:
history = model.fit(X_train_scaled,y_train_cat,callbacks=[es],
                    epochs = 300,batch_size= 256, validation_split = 0.2,verbose = 1)

Epoch 1/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8050 - loss: 0.5302 - val_accuracy: 0.7802 - val_loss: 0.6384
Epoch 2/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8051 - loss: 0.5296 - val_accuracy: 0.7831 - val_loss: 0.6246
Epoch 3/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8048 - loss: 0.5320 - val_accuracy: 0.7798 - val_loss: 0.6322
Epoch 4/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8050 - loss: 0.5288 - val_accuracy: 0.7817 - val_loss: 0.6286
Epoch 5/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8043 - loss: 0.5300 - val_accuracy: 0.7809 - val_loss: 0.6287
Epoch 6/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8045 - loss: 0.5304 - val_accuracy: 0.7813 - val_loss: 0.6304
Epoch 7/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8048 - loss: 0.5291 - val_accuracy: 0.7810 - val_loss: 0.6403
Epoch 8/300
821/821 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8049 - loss: 0.5293 - val_accu

In [54]:
nn_test_loss, nn_test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=1)
print('NN test accuracy:', nn_test_acc)

15750/15750 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.6931 - loss: 1.4769
NN test accuracy: 0.6931448578834534


In [50]:
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train_scaled, y_train)
print('RF test accuracy:', accuracy_score(y_test, rf.predict(X_test_scaled)))

RF test accuracy: 0.6579940476190477


In [56]:
import joblib
#joblib.dump(rf, '/content/drive/MyDrive/Sensornet.ai/models/rf_model.pkl')
model.save('/content/drive/MyDrive/Sensornet.ai/models/dense_model.keras')
joblib.dump(scaler, '/content/drive/MyDrive/Sensornet.ai/models/scaler.pkl')

['/content/drive/MyDrive/Sensornet.ai/models/scaler.pkl']

This project compared a dense neural network and a Random Forest classifier for fault diagnosis on the Tennessee Eastman Process, using a 5% sample of the data. The 52 process variables were used as inputs to classify 21 classes (normal operation plus 20 fault types).

The neural network (two hidden layers of 128 and 64 neurons, trained with early stopping) reached about 80% training and 78% validation accuracy, and 69.3% accuracy on the held-out test set. The Random Forest, using default hyperparameters with 200 trees, reached 65.8% test accuracy. The neural network therefore outperformed the Random Forest by about 3.5 percentage points, suggesting it captures the relationships between the process variables somewhat better than a tree ensemble on this data.

Both models scored lower on the test set than on validation. This is expected for the TEP benchmark: test runs are longer than training runs and introduce faults much later (about 8 hours in, compared with about 1 hour in training), so a large share of each test run is normal operation that carries a fault label, which limits the accuracy any model can reach. In addition, faults 3, 9 and 15 are known to be very hard to detect from the measured variables, which lowers recall for all models.

Future work could include hyperparameter tuning, windowed time-series inputs with an LSTM, training on the full dataset instead of a 5% sample, and dedicated methods for the hard-to-detect faults.